In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import squarify
import sys
sys.path.append('../../src')
from data_imports import *
from seaborn_helper_functions import *
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
patients = import_patients()
biosamples = import_biosamples()
amplicons = import_amplicons()
genes = import_genes()

## Cohort treemap

Two-level treemap of the patient cohort. Outer blocks are `cancer_type`; inner cells are `cancer_subclass` (patients with no subclass annotation are pooled into an `(unspecified)` cell within their type). Cell area is proportional to patient count.

In [ ]:
def nested_treemap(df, level1, level2, ax=None, width=30, height=20,
                   pad_type=0.0, min_label_area=0, l2_label_area=0.5,
                   fontsize_type=7, fontsize_sub=7):
    """Two-level treemap. Outer cells = level1, inner cells = level2.
    Each level1 group gets one color; level2 cells within share it, split by white borders.
    Cell area is proportional to the number of rows (patients) in each group."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 9))

    # level1 counts, sorted descending so squarify lays them big -> small
    l1_counts = df[level1].value_counts()
    l1_order = l1_counts.index.tolist()

    # color per level1: concatenate tab20/b/c for up to 60 distinct colors
    palette = [c for cm in ("tab20", "tab20b", "tab20c")
               for c in plt.get_cmap(cm).colors]
    colors = {name: palette[i % len(palette)] for i, name in enumerate(l1_order)}

    # outer layout
    l1_norm = squarify.normalize_sizes(l1_counts.values, width, height)
    l1_rects = squarify.squarify(l1_norm, 0, 0, width, height)

    for name, rect in zip(l1_order, l1_rects):
        x, y, dx, dy = rect["x"], rect["y"], rect["dx"], rect["dy"]
        # inner: subclass counts within this cancer_type (NA -> "(unspecified)")
        sub = df.loc[df[level1] == name, level2].fillna("NOS").value_counts()
        # shrink the region slightly to leave a gutter between level1 blocks
        ix, iy = x + pad_type, y + pad_type
        idx, idy = max(dx - 2 * pad_type, 1e-6), max(dy - 2 * pad_type, 1e-6)
        sub_norm = squarify.normalize_sizes(sub.values, idx, idy)
        sub_rects = squarify.squarify(sub_norm, ix, iy, idx, idy)
        for (sname, scount), srect in zip(sub.items(), sub_rects):
            ax.add_patch(mpatches.Rectangle(
                (srect["x"], srect["y"]), srect["dx"], srect["dy"],
                facecolor=colors[name], edgecolor="white", linewidth=0.6))
            # label large subclass cells; break on '_' so long names render more square
            if srect["dx"] * srect["dy"] >= l2_label_area:
                ax.text(srect["x"] + srect["dx"] / 2, srect["y"] + srect["dy"] / 2,
                        f"{sname.replace('_', chr(10))}\n({scount})", ha="center", va="center",
                        fontsize=fontsize_sub, color="white", alpha=0.85)
        # outline the whole level1 block
        ax.add_patch(mpatches.Rectangle((x, y), dx, dy, fill=False,
                                        edgecolor="0.15", linewidth=1.3))
        # label the level1 block along its bottom edge
        if dx * dy >= min_label_area:
            ax.text(x + dx / 2, y + dy / 2, f"{name} ({l1_counts[name]})",
                    ha="center", va="top", fontsize=fontsize_type,
                    fontweight="bold", color="0.1")

    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_aspect("equal")
    return (fig,ax)

In [ ]:
fig,ax = nested_treemap(patients, "cancer_type", "cancer_subclass", width=40, height=20)
ax.set_title(f"Pediatric pan-cancer cohort (n={len(patients)} patients)\n"
             "outer blocks: tumor type   |   inner cells: molecular subclass",
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
savefig(fig,'out/treemap_patients')

In [ ]:
patients[~patients.OS_status.isna() & ~patients.OS_months.isna()]

In [ ]:
surv = patients[~patients.OS_status.isna() & ~patients.OS_months.isna()]
fig,ax = nested_treemap(surv, "cancer_type", "cancer_subclass", width=30, height=12)
ax.set_title(f"pedpancan cases with available survival data (n={len(surv)} patients)\n"
             "outer blocks: tumor type   |   inner cells: molecular subclass",
             fontsize=11)
plt.tight_layout()
plt.show()
savefig(fig,'out/treemap_survival')